# Project 1 Code Pipeline

In [7]:
!pip install duckdb pandas pyarrow scikit-learn imbalanced-learn xgboost lightgbm bayesian-optimization matplotlib seaborn requests

## Imports and Logging

In [8]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import io
import requests

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (roc_auc_score, confusion_matrix, 
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from bayes_opt import BayesianOptimization

# Logging
logging.basicConfig(
    filename='pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
log = logging.getLogger(__name__)
log.info("Pipeline started")
print("Imports successful")

Imports successful


## Load Files

In [9]:
Urls = {
    'transaction': 'https://myuva-my.sharepoint.com/:u:/g/personal/bqu3tr_virginia_edu/IQA0m477oIf-RKPajvdNqdR5AXpsLX6vtr8NlrCK4-5UzLE?download=1',
    'cards': 'https://myuva-my.sharepoint.com/:u:/g/personal/bqu3tr_virginia_edu/IQC7ytwes3e9Q4THqdTdyLl3ASwh6O6OHBHvzi4SDoc7g1k?download=1',
    'email': 'https://myuva-my.sharepoint.com/:u:/g/personal/bqu3tr_virginia_edu/IQCbzr0lDS4yTJTAEQxTZaFqAWBKuDSTrK4H6siEgs5Fw_8?download=1',
    'identity': 'https://myuva-my.sharepoint.com/:u:/g/personal/bqu3tr_virginia_edu/IQCtneexJFggSr_cdos1vysfAca77zSoaBFJqXNQ5-0xcWU?download=1'
}

def load_parquet_from_onedrive(url: str, name: str) -> pd.DataFrame:
    """Download a parquet file from a OneDrive sharing link."""
    try:
        log.info(f"Downloading {name} from OneDrive...")
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=120)
        response.raise_for_status()
        df = pd.read_parquet(io.BytesIO(response.content))
        log.info(f"{name} loaded: {df.shape}")
        return df
    except requests.exceptions.RequestException as e:
        log.error(f"Failed to download {name}: {e}")
        raise
    except Exception as e:
        log.error(f"Failed to read parquet for {name}: {e}")
        raise

dfs = {name: load_parquet_from_onedrive(url, name) for name, url in Urls.items()}

print("Tables Loaded:")
for name, df in dfs.items():
    print(f"{name}: {df.shape}")

Tables Loaded:
transaction: (1097231, 344)
cards: (1097231, 9)
email: (1097231, 12)
identity: (286140, 79)


## DuckDB SQL Query

In [10]:
try: 
    con = duckdb.connect()

    for name, df in dfs.items():
        con.register(name, df)
        log.info(f"Registered {name} in DuckDB")
    
    query = """
        SELECT
            t.TransactionID,
            t.isFraud,
            t.TransactionAmt,
            t.TransactionDT,
            t.ProductCD,
            c.card1, c.card2, c.card3, c.card4, c.card5, c.card6,
            c.addr1, c.addr2,
            e.P_emaildomain, e.R_emaildomain,
            e.M1, e.M2, e.M3, e.M4, e.M5, e.M6, e.M7, e.M8, e.M9,
            i.DeviceType, i.DeviceInfo,
            i.id_01, i.id_02, i.id_03, i.id_04, i.id_05,
            i.id_06, i.id_07, i.id_08, i.id_09, i.id_10
        FROM transaction t
        LEFT JOIN cards c ON t.TransactionID = c.TransactionID
        LEFT JOIN email e ON t.TransactionID = e.TransactionID
        LEFT JOIN identity i ON t.TransactionID = i.TransactionID
    """

    df = con.execute(query).fetchdf()
    log.info(f"Joined DataFrame shape: {df.shape}")

except Exception as e:
    log.error(f"Error during DuckDB operations: {e}")
    raise
finally:
    con.close()

## Feature Engineering

In [11]:
try:
    # Encode categorical features
    cat_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain','R_emaildomain', 'DeviceType', 'DeviceInfo',
                'M1','M2','M3','M4','M5','M6','M7','M8','M9']
    
    le = LabelEncoder()
    for col in cat_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))
            log.info(f"Encoded {col}")
    
    # Fill missing numeric with median
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    num_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    log.info("Filled missing numeric values with median")

    # Drop missing isFraud
    df = df.dropna(subset=['isFraud'])
    log.info(f"DataFrame shape after dropping missing target: {df.shape}")

    # Separate features and target
    X = df.drop(columns=['TransactionID', 'isFraud'])
    y = df['isFraud']

    print(f"Features shape: {X.shape}, Target shape: {y.shape}")
    
except Exception as e:
    log.error(f"Error during preprocessing: {e}")
    raise    


Features shape: (590540, 34), Target shape: (590540,)


## Train Test Split

In [12]:
try: 
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    log.info(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

    # Apply SMOTE to the training data
    smote = SMOTE(random_state=42)
    X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
    log.info(f"After SMOTE - Train shape: {X_train_sm.shape}, Target distribution: {np.bincount(y_train_sm)}")
    print(f"After SMOTE - Train shape: {X_train_sm.shape}, Target distribution: {np.bincount(y_train_sm)}")
except Exception as e:
    log.error(f"Error during train-test split or SMOTE: {e}")
    raise

Train shape: (472432, 34), Test shape: (118108, 34)
After SMOTE - Train shape: (911804, 34), Target distribution: [455902 455902]


/tmp/ipykernel_10283/3130091131.py:9: DeprecationWarning: Non-integer input passed to bincount. In a future version of NumPy, this will be an error. (Deprecated NumPy 2.1)
  log.info(f"After SMOTE - Train shape: {X_train_sm.shape}, Target distribution: {np.bincount(y_train_sm)}")
/tmp/ipykernel_10283/3130091131.py:10: DeprecationWarning: Non-integer input passed to bincount. In a future version of NumPy, this will be an error. (Deprecated NumPy 2.1)
  print(f"After SMOTE - Train shape: {X_train_sm.shape}, Target distribution: {np.bincount(y_train_sm)}")


## Bayesian Optimization

In [ ]:
try:
    # XGBoost
    def xgb_eval(n_estimators, max_depth, learning_rate, subsample):
        model = XGBClassifier(
            n_estimators=int(n_estimators),
            max_depth=int(max_depth),
            learning_rate=learning_rate,
            subsample=subsample,
            random_state=42,
            eval_metric='auc',
            verbosity=0
        )
        return cross_val_score(model, X_train_sm, y_train_sm, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    
    xgb_bo = BayesianOptimization(
        f=xgb_eval,
        pbounds={
            'n_estimators': (100, 500),
            'max_depth': (3, 8),
            'learning_rate': (0.01, 0.3),
            'subsample': (0.6, 1.0)
        },
        random_state=42
    )
    xgb_bo.maximize(init_points=5, n_iter=10)
    log.info(f"XGBoost best params: {xgb_bo.max['params']}")
    print(f"XGBoost best params: {xgb_bo.max['params']}")
except Exception as e:
    log.error(f"Error during XGBoost optimization: {e}")
    raise